In [ ]:
pip install openai PyPDF2

In [ ]:
from google.colab import files
uploaded = files.upload()
print(uploaded)
file_name = list(uploaded.keys())[0]


In [ ]:
import PyPDF2
def read_resume(file_path):
   text = ""
   with open(file_path, "rb") as file:
       reader = PyPDF2.PdfReader(file)
       for page in reader.pages:
           text += page.extract_text()
   return text
print(read_resume(file_name))

In [ ]:
from google.colab import userdata

In [ ]:
pip install google-genai

In [ ]:
from google import genai
client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

In [ ]:
def analyze_resume(text):
   response = client.models.generate_content(
       model="gemini-2.5-flash-lite",
       contents=f"""
You are an ATS resume analyzer.

Analyze the resume below and return ATS score and feedback.

Resume:
{text}

Return in JSON format with:
- ats_score (0-100)
- summary
- strengths
- weaknesses

Rules:
- Output ONLY JSON
- No extra text
"""
   )
   return response.text


In [ ]:
resume_text = read_resume(file_name)
result = analyze_resume(resume_text)
print(result)

In [ ]:
import json

clean_result = result.replace("```json", "").replace("```", "")

data = json.loads(clean_result)

In [ ]:
from IPython.display import HTML, display

html_code = f"""

<style>

body {{
    font-family: Arial, sans-serif;
    background: #f4f7fb;
}}

.container {{
    max-width: 900px;
    margin: auto;
    padding: 20px;
}}

.title {{
    text-align: center;
    font-size: 42px;
    color: #2E86C1;
    margin-bottom: 30px;
    font-weight: bold;
}}

.card {{
    background: white;
    border-radius: 20px;
    padding: 25px;
    margin-top: 25px;
    box-shadow: 0px 6px 20px rgba(0,0,0,0.1);
}}

.score-container {{
    display: flex;
    justify-content: center;
    align-items: center;
    flex-direction: column;
}}

.circular-chart {{
    width: 220px;
    height: 220px;
}}

.circle-bg {{
    fill: none;
    stroke: #eee;
    stroke-width: 3.8;
}}

.circle {{
    fill: none;
    stroke-width: 3.8;
    stroke-linecap: round;
    stroke: #28B463;
    animation: progress 1.5s ease-out forwards;
}}

@keyframes progress {{
    0% {{
        stroke-dasharray: 0 100;
    }}
}}

.percentage {{
    fill: #145A32;
    font-size: 0.5em;
    text-anchor: middle;
    font-weight: bold;
}}

.section-title {{
    color: #1B4F72;
    margin-bottom: 15px;
    font-size: 28px;
}}

.summary {{
    font-size: 18px;
    line-height: 1.8;
    color: #333;
}}

.strengths {{
    background: #EAFAF1;
}}

.weaknesses {{
    background: #FDEDEC;
}}

ul {{
    padding-left: 20px;
}}

li {{
    margin-bottom: 12px;
    font-size: 17px;
    color: #333;
}}

</style>


<div class="container">

    <div class="title">
        ATS Resume Analyzer
    </div>

    <div class="card">

        <div class="score-container">

            <h2 class="section-title">
                ATS Score
            </h2>

            <svg viewBox="0 0 36 36" class="circular-chart">

                <path class="circle-bg"
                    d="M18 2.0845
                    a 15.9155 15.9155 0 0 1 0 31.831
                    a 15.9155 15.9155 0 0 1 0 -31.831"
                />

                <path class="circle"
                    stroke-dasharray="{data['ats_score']}, 100"
                    d="M18 2.0845
                    a 15.9155 15.9155 0 0 1 0 31.831
                    a 15.9155 15.9155 0 0 1 0 -31.831"
                />

                <text x="18" y="20.35" class="percentage">
                    {data['ats_score']}%
                </text>

            </svg>

        </div>

    </div>


    <div class="card">

        <h2 class="section-title">
            Professional Summary
        </h2>

        <div class="summary">
            {data['summary']}
        </div>

    </div>


    <div class="card strengths">

        <h2 class="section-title">
            Strengths
        </h2>

        <ul>
            {''.join(f"<li>{item}</li>" for item in data['strengths'])}
        </ul>

    </div>


    <div class="card weaknesses">

        <h2 class="section-title">
            Weaknesses
        </h2>

        <ul>
            {''.join(f"<li>{item}</li>" for item in data['weaknesses'])}
        </ul>

    </div>

</div>

"""

In [ ]:
display(HTML(html_code))